# Experiment 01 — Zero-Shot Baseline

**Model:** `google/mt5-small`  
**Goal:** Establish a floor ROUGE score with no fine-tuning.  
**Expected:** Low scores — this is the starting reference point only.

## Install dependencies

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate rouge-score

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/health_qa/'
SAVE_DIR = '/content/drive/MyDrive/health_qa/outputs/'

import os
os.makedirs(SAVE_DIR, exist_ok=True)

## Imports and seed

In [ ]:
import random, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Load data

In [ ]:
train_df = pd.read_csv(DATA_DIR + 'Train.csv')
val_df   = pd.read_csv(DATA_DIR + 'Val.csv')
test_df  = pd.read_csv(DATA_DIR + 'Test.csv')

print(f'Train : {train_df.shape}')
print(f'Val   : {val_df.shape}')
print(f'Test  : {test_df.shape}')
train_df.head(3)

## Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

train_df['language'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Examples per language')
axes[0].set_xlabel('Language')
axes[0].tick_params(axis='x', rotation=30)

train_df['country'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Examples per country')
axes[1].tick_params(axis='x', rotation=30)

train_df['resp_len'] = train_df['response'].str.split().str.len()
train_df['resp_len'].hist(bins=50, ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Response length distribution (words)')
axes[2].axvline(train_df['resp_len'].median(), color='red', linestyle='--',
                label=f"Median: {train_df['resp_len'].median():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig(SAVE_DIR + 'eda_overview.png', dpi=150)
plt.show()

print('Median response length:', train_df['resp_len'].median())
print('95th percentile:', train_df['resp_len'].quantile(0.95))

## Language × country cross-tab

In [ ]:
pd.crosstab(train_df['language'], train_df['country'])

## Clean text and build prompts

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)
    return re.sub(r' {2,}', ' ', text).strip()

def build_prompt(question, language, country=''):
    header = f'Language: {str(language).capitalize()}'
    if country:
        header += f' | Country: {country}'
    return f'{header}\nQuestion: {question}\nAnswer:'

for df in [train_df, val_df, test_df]:
    df['prompt']      = df['prompt'].apply(clean_text)
    df['language']    = df['language'].str.strip().str.lower()
    df['country']     = df['country'].str.strip()
    df['prompt_text'] = df.apply(
        lambda r: build_prompt(r['prompt'], r['language'], r['country']), axis=1
    )

for df in [train_df, val_df]:
    df['response'] = df['response'].apply(clean_text)

print(train_df['prompt_text'].iloc[0])
print()
print(train_df['response'].iloc[0])

## Load model (zero-shot — no training)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = 'google/mt5-small'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print('Loaded.')

## Batch generation helper

In [ ]:
def generate_batch(prompts, batch_size=8, max_new_tokens=200):
    model.eval()
    predictions = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch  = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
        predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return predictions

## ROUGE evaluation helper

In [ ]:
_scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def evaluate_rouge(predictions, references):
    r1s, rls = [], []
    for pred, ref in zip(predictions, references):
        s = _scorer.score(ref, pred)
        r1s.append(s['rouge1'].fmeasure)
        rls.append(s['rougeL'].fmeasure)
    return {
        'rouge1': round(sum(r1s) / len(r1s), 4),
        'rougeL': round(sum(rls) / len(rls), 4),
    }

## Validate on 200-example sample

In [ ]:
val_sample = val_df.sample(200, random_state=SEED).reset_index(drop=True)
val_preds  = generate_batch(val_sample['prompt_text'].tolist())
scores     = evaluate_rouge(val_preds, val_sample['response'].tolist())

print(f"ROUGE-1 F1 : {scores['rouge1']}")
print(f"ROUGE-L F1 : {scores['rougeL']}")
print(f"Est. LB    : {0.37*scores['rouge1'] + 0.37*scores['rougeL']:.4f}")

## Generate test predictions and build submission CSV

In [ ]:
test_preds = generate_batch(test_df['prompt_text'].tolist())

submission = pd.DataFrame({
    'ID':         test_df['ID'].values,
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds,
})

out_path = SAVE_DIR + 'exp01_zero_shot_baseline.csv'
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
submission.head(3)

## Results

| Metric | Score |
|--------|-------|
| ROUGE-1 F1 | *(fill after run)* |
| ROUGE-L F1 | *(fill after run)* |
| Zindi LB score | *(fill after submission)* |

**Insight:** Zero-shot mT5-small produces very low scores. Fine-tuning on the training data is essential. Proceeding to Experiment 03.